## 7. 大模型增强与集成
大语言模型的认知范围和能力范围都是有限的，认知范围取决于训练时的数据集，而能力范围则仅限于文本生成。本章就是要介绍如何通过增强与集成技术克服大模型的这些局限，同时会介绍具体的实现框架LangChain，但该框架在大模型的调用上需要使用第三方提供的接口，所以读者若想运行示例代码，需要先在第三方平台获取接口的调用token。这一章的内容其实很关键，对于想从事人工智能开发的人来说，这是成本较低的介入方向。

### 7.1  检索增强生成
检索增强生成（Retrieval Augmented Generation，以下简称RAG）是一种结合了信息检索和语言生成的全新技术，它是解决大语言模型认知局限性的重要方法，可极大提升自然语言处理结果的相关性和准确性。RAG的基本思想就是引入外部知识源来扩展模型眼界，使其可以在生成响应时参考这些额外信息，本质上是检索技术与大模型提示工程相结合的产物。

![RAG模型结构](./images/rag-model.png)

#### 7.1.1  从关键字检索到向量检索
这一小节主要介绍在大语言模型背景下，检索技术也由原来的关键字检索升级为向量检索，向量检索是整个RAG技术的关键一环。早期向量检索使用关键字出现频率构造向量，比如TF-IDF（Term Frequency-Inverse Document Frequency）、BM25（Best Matching 25）等。这类方法构造出来的片段向量通常是一个稀疏向量（Sparse Vector），所以它们所对应的检索方法也被称为稀疏片段检索（Sparse Passage Retrieval，SPR）。转换器架构的仅编码器模型最擅长的正是将文本转换为稠密向量，所以只要对仅编码器模型稍做调整，它就可以用于文档到片段向量的转换。与稀疏片段检索相对应，这种基于稠密向量的检索方法被称为稠密片段检索（Dense Passage Retrieval，以下简称DPR）。

#### 7.1.2  实现DPR
transformers库提供了DPRContextEncoder和DPRContextEncoderTokenizer，代表的是上下文编码器及其分词器，而DPRQuestionEncoder和DPRQuestionEncoderTokenizer则是查询编码器及其分词器。这些类也提供了from_pretrained方法，可以将预训练的模型和分词器直接加载进来做向量检索。如示例7.1所示：

In [ ]:
from transformers import (
    DPRQuestionEncoder, DPRQuestionEncoderTokenizer,
    DPRContextEncoder, DPRContextEncoderTokenizer
)
import torch

# 初始化文档编码器及其分词器
ctx_enc = "facebook/dpr-ctx_encoder-single-nq-base"
ctx_tokenizer = DPRContextEncoderTokenizer.from_pretrained(ctx_enc)
ctx_encoder = DPRContextEncoder.from_pretrained(ctx_enc)

# 初始化查询编码器及其分词器
q_enc = "facebook/dpr-question_encoder-single-nq-base"
q_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained(q_enc)
q_encoder = DPRQuestionEncoder.from_pretrained(q_enc)

# 示例文档列表
documents = [
    "DPR(Dense Passage Retrieval) is a method for information retrieval.",
    "The transformers library provides pre-trained models for NLP tasks.",
    "Faiss is a library for similarity search and clustering of vectors."
]

# 将文档编码为向量
def encode_contexts(contexts):
    inputs = ctx_tokenizer(contexts, return_tensors='pt',\
                     padding=True, truncation=True, max_length=512)
    with torch.no_grad():  # 禁用梯度计算
        embeddings = ctx_encoder(**inputs).pooler_output
    return embeddings  # 返回 PyTorch 张量

# 将查询编码为向量
def encode_question(question):
    inputs = q_tokenizer(question, return_tensors='pt',\
                                    truncation=True, max_length=512)
    with torch.no_grad():  # 禁用梯度计算
        embeddings = q_encoder(**inputs).pooler_output
    return embeddings  # 返回 PyTorch 张量

# 计算内积相似度
def inner_product_similarity(query_embedding, ctx_embeddings):
    query_embedding = query_embedding.squeeze()  # 去掉批次维度
    # 计算所有文档向量与查询向量的内积
    similarities = torch.matmul(ctx_embeddings, query_embedding)
    return similarities.tolist()
# 创建文档向量
ctx_embeddings = encode_contexts(documents)
# 执行检索
query = "What is DPR?"
query_embedding = encode_question(query)
# 计算相似度
similarities = inner_product_similarity(query_embedding, ctx_embeddings)
# 获取最相关的文档
top_k = 3
tops = sorted(range(len(similarities)),\
               key=lambda i: similarities[i], reverse=True)[:top_k]
# 输出检索结果
for i in tops:
    print(f"Doc{i}: {documents[i]} (Similarity: {similarities[i]:.4f})")

#### 7.1.3  向量存储
向量存储是将文档预先转换为向量，并将文档及其对应的向量一并保存起来。因为向量本身并不是用户检索所需要的结果，它的存在只是为了辅助文档与查询之间的相似度计算。所以预先将文档转换成向量的根本目的，其实是建立文档基于向量的索引。在实际开发中并不需要从头开始建立文档的向量索引，支持为文档建立向量索引的开源库有很多，比如Annoy（Approximate Nearest Neighbors Oh Yeah）、FAISS（Facebook AI Similarity Search）等等。下面的代码使用FAISS建立向量的平面索引并保存到文件中：

In [ ]:
# 创建索引
index = faiss.IndexFlatIP(768)  # 768是DPR模型输出的向量维度
ctx_embeddings = encode_contexts(documents)
index.add(ctx_embeddings)
faiss.write_index(index, "faiss_index/dpr.idx")

下面的代码则使用保存的索引进行了检索：

In [ ]:
# 执行检索
index = faiss.read_index("faiss_index/dpr.idx")
query = "What is DPR?"
q_embedding = encode_question(query)
distances, indices = index.search(q_embedding, k=3) # k是最相关文档数量
# 输出检索结果
for idx, distance in zip(indices[0], distances[0]):
    print(f"Document {idx}: {documents[idx]} ({distance})")

#### 7.1.4  增强生成
transformers库提供了基于RAG原论文实现的检索器和生成器，通过它们可以实现检索增强：

In [ ]:
from transformers import (
    RagTokenizer, RagRetriever,
    RagSequenceForGeneration
)
model_id = "facebook/rag-sequence-nq"
tokenizer = RagTokenizer.from_pretrained(model_id)
retriever = RagRetriever.from_pretrained(model_id, index_name="exact",
                                         use_dummy_dataset=True)
model = RagSequenceForGeneration.from_pretrained(model_id,
                                         retriever=retriever)
q = "how many countries are in europe"
input_dict = tokenizer.prepare_seq2seq_batch(q, return_tensors="pt")
generated = model.generate(input_ids=input_dict["input_ids"])
print(tokenizer.batch_decode(generated, skip_special_tokens=True)[0])

基于RAG原论文实现的检索增强生成，其生成器采用的大模型是与BART模型绑定在一起的。而在实际应用中，往往需要根据实际情况切换成不同的大模型。此外，RAG检索器使用维基百科作为外部知识库，但这并不一定是实际应用中需要的外部知识库。为了增强RAG系统的灵活性，人们对其结构做了一些调整，现在实际应用中RAG系统的结构大致如下图所示：

![RAG结构](./images/rag.png)

### 7.2  逻辑推理引擎
这一小节介绍了大模型的逻辑推理能力及其具体应用。

#### 7.2.1  思维链提示
大模型在解决复杂问题时的表现不好，主要集中在需要数学推理（Arithmetic Reasoning）、常识推理（Commonsense Reasoning）以及符号推理（Symbolic Reasoning）才能解决的多步骤任务上。这主要是因为大语言模型本质上仍然是预测模型，它在回复用户输入时并不是真的在解决问题，而是通过概率来补齐用户输入的后续文本。大语言模型之所以不能得到正确的结果，就是因为缺少了中间构建思维链的过程。如果能够引导大模型像人类一样先构建思维链，将复杂问题拆解成模型认知范围内的简单问题，那么大模型就能够通过推理逐步完成复杂任务。具体方式就是在提示中添加一些构建思维链的提示，具体请参阅本小节书中内容。

#### 7.2.2  程序辅助语言模型
思维链提示打通了大语言模型处理复杂问题的瓶颈，但它并不能从根本上解决大模型在数学运算上的问题。程序辅助语言模型（Program-Aided Language model，以下简称PAL）借助提示工程让大模型生成可执行的Python代码，并通过外部解释器执行代码来弥补大模型在数学上的能力缺陷。但大模型本身并没有调用Python解释器的能力，执行大模型输出的Python脚本，要么由人手工粘贴完成，要么开发专门的程序自动完成。显然在实际应用中，开发专门的应用程序将整个流程自动化是更为高效的做法。下图展示了自动化PAL的基本结构图：

![PAL结构](./images/pal.png)

详细内容请参阅本小节书中描述。

#### 7.2.3  推理动作模型
推理动作语言模型（Reasoning Acting，ReAct）是当今大火的AI Agent的理论基础，读懂这一部分就能理解AI Agent的运行机制。ReAct在结构上与RAG、PAL类似，只不过它可以调用的资源不再是单纯知识库和Python解释器，而是任意一种可调用资源，比如模型、数据库、API等等。具体原理请参阅本小节书中内容。

#### 7.2.4  Agent与MCP
介绍了基于ReAct的Agent与MCP技术，详见书中内容。

### 7.3  使用LangChain实现RAG
LangChain是一个专门用于集成大语言模型的开源框架，通过这个框架可以轻松实现RAG、PAL、ReAct等大模型逻辑推理能力。

#### 7.3.1  链与Runnable接口
LangChain以链式结构处理任务，每个业务结点完成对任务处理的一个步骤，下图展示了一个链的例子：

![链结构](./images/chain.png)

这是一个通过提示增强的智能任务处理链，大语言模型是最核心的组件。但需要注意的是，LangChain并不会像transformers库那样实际加载模型，而是调用第三方发布的公共大语言模型服务。这就是本书第1章中介绍的应用模型的第二种方法，即通过大模型发布的服务接口调用大模型。下面代码展示了基于Azure Open AI模型构建的问答链：

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个知识丰富的AI助手。"),
    ("user", "{question}"),
])
llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_deployment=os.environ["OPENAI_MODEL_NAME_LLM"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ["OPENAI_API_VERSION_LLM"],
)
parser = StrOutputParser()

chain = prompt | llm | parser

result = chain.invoke({"question": "美国2020年的总统是谁?"})
print(result)

LangChain链的组件都实现了Runnable接口，LangChain链本身也可以看成是Runnable接口的实例。所以Runnable接口才是LangChain链的本质，它定义了LangChain链及其组件的全部执行方法，具体方法及其使用方法见书中介绍。

#### 7.3.2 文档加载与向量存储
LangChain为了抽象文档资源及其加载方法，专门定义了Document和BaseLoader来实现文档的向量化。下面的代码是以向量化PDF文件为实例，采用FAISS框架将向量化后文档和索引保存到本地文件中：

In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import AzureOpenAIEmbeddings

# 加载嵌入模型
load_dotenv()
embeddings = AzureOpenAIEmbeddings(
    model=os.environ["OPENAI_MODEL_NAME_EMBEDDING"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    openai_api_version=os.environ["OPENAI_API_VERSION_EMBEDDING"]
)
# 加载PDF文档
loader = PyPDFLoader('/path/to/your/doc.pdf')
document = loader.load()
# 文本拆分
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(document)
# 生成嵌入向量并保存
vs = FAISS.from_documents(texts, embeddings)
vs.save_local("/path/to/faiss_index/")

#### 7.3.3 构建RAG链
LangChain定义了StuffDocumentsChain来代表在提示中添加文档的处理链，构建StuffDocumentsChain一般可通过函数create_stuff_documents_chain来实现，下面的代码展示了这个构建过程及其使用方法：

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langchain_core.documents import Document
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

# 初始化模型
load_dotenv()
chat_model = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_deployment=os.environ["OPENAI_MODEL_NAME_LLM"],
    api_version=os.environ["OPENAI_API_VERSION_LLM"],
)
# 定义 Prompt
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template='''
    Based on the context below, answer the question:
    Context: {context}
    Question: {question}
    '''
)
# 创建StuffDocumentsChain
chain = create_stuff_documents_chain(llm=chat_model, prompt=prompt)
# 示例文档
documents = [
    Document(page_content="这是一本关于生成式人工智能的书籍，它是......"),
    Document(page_content="人工智能包括机器学习、神经网络......等。")
]
# 提问
question = "What is the book about?"
# 执行链
result = chain.invoke({"context": documents, "question": question})
print(result)

StuffDocumentsChain可以看成是RAG中的生成器，只要再将它与VectorStoreRetriver结合成新的链就可以处理RAG流程了。LangChain中的函数create_retrieval_chain可用于构建这样的RAG链，下面的代码展示了加载向量存储、创建StuffDocumentsChain，最终构建RAG链的完整过程：

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

# 加载大模型
load_dotenv()
embeddings = AzureOpenAIEmbeddings(
    model=os.environ["OPENAI_MODEL_NAME_EMBEDDING"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    openai_api_version=os.environ["OPENAI_API_VERSION_EMBEDDING"],
    openai_api_type="azure",
)
llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_deployment=os.environ["OPENAI_MODEL_NAME_LLM"], 
    api_version=os.environ["OPENAI_API_VERSION_LLM"],
)

# 加载向量数据库
db = FAISS.load_local("/path/to/faiss_index",  embeddings)
retriever = db.as_retriever()
# 创建提示模板
prompt = PromptTemplate(
    input_variables=["context", "input"],
    template='''
    Based on the context below, answer the question:
    Context: {context}
    Question: {input}
    '''
)

# 创建StuffDocumentsChain
stuff_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
# 创建RAG链
qa = create_retrieval_chain(retriever, stuff_chain)
# 执行RAG链
query = "your question here?"
result = qa.invoke({"input":query})
print(result)

### 7.4  代理与LangGraph*
这一节主要介绍LangChain中的Agent，以及对Agent的扩展LangGraph，它们的设计都是为了实现ReAct模型，也是当前所谓智能Agent的LangChain解决方案。LangChain有两种方式可以实现类似ReAct这样的复杂流程，一种就是LangChain中传统的代理（Agent）模式，而另一种则是LangChain的扩展框架LangGraph。如果将LangChain链看成是事先定义好的静态执行流程，那么代理则可依赖大模型的推理能力，部分或完全自主地动态构建执行流程。LangGraph可以看成是对代理的进一步扩展，它将代理底层机制做了进一步的泛化，将链式执行流程拓展到了基于图的执行流程。图由节点（Node）和边（Edge）组成，通过图可以构建更为复杂的网状结构，这其中当然也包含ReAct中的环状结构。LangGraph是对代理本质的进一步抽象，未来构建复杂流程的主要方式应该会集中在LangGraph上。有关它们的具体使用方法，请参阅书中本小节中的介绍。

### 7.5  本章小结
本章介绍了增强大模型的几种技术，包括检索增强生成RAG、思维链CoT、程序辅助语言PAL和推理动作ReAct等。这些增强技术都需要通过居间协调的程序实现增强，而LangChain正是实现这种居间功能的框架。
RAG是将大语言模型与外部知识系统集成起来的技术，通过这种技术可以解决大模型知识截止和幻觉等问题，同时也将大模型的认知范围扩展到了无限的空间。RAG一般由检索器和生成器两部分组成，检索器通常采用向量检索的方式实现，而生成器则需要借助大模型的语言生成能力实现。RAG相当于是给大模型添加了“五官”，让大模型的眼界扩展到了无限的知识空间。
CoT是激发大模型逻辑推理能力的技术，可以显著提升模型解决复杂问题的能力。CoT分为有样本提示和零样本提示两种，它们都可以在一定程度上引导模型拆解任务。目前，许多AI助手已经将思维链引入问答流程中了，这不仅可以提高模型解答问题的准确性，也提升了模型给出结论的可解释性。但CoT只是激活了大模型在逻辑上拆解复杂任务的能力，还需借助PAL和ReAct技术去实际执行任务。PAL和ReAct是通过与外部系统集成起来的方式，彻底破除了大模型在执行能力上的限制。前者是将大模型与Python解释器集成起来，通过生成可执行的代码来提升模型的数学运算能力；而后者则将模型与任意外部系统集成起来，让模型拥有了控制一切的能力。PAL和ReAct就像是为大模型添加上了四肢，从此大模型的能力就不再仅限简单的问答或文本生成了。
LangChain和LangGraph是实现以上技术的重要框架，前者适用于与大模型的交互只有一次的流程，而后者则适用于需要多次交互，并在多次交互中需要共享数据的应用场景。通过LangChain和LangGraph可轻松实现RAG和ReAct，它们将是未来生成式AI开发中主要的应用场景之一。